# Model Training - Smart Grid Security
**Kelompok 3 | KDA Project**

Pipeline training model klasifikasi untuk deteksi anomali pada Smart Grid.

| Label | Keterangan |
|-------|------------|
| 0     | Normal     |
| 1     | Attack     |
| 2     | Fault      |

**Model:** Decision Tree | Random Forest | Logistic Regression  
**Strategi:** Multi-iterasi (5x) dengan averaging probabilitas prediksi  
**Output:** `test_predictions.csv`, `model_summary.csv`, `iteration_metrics.csv`

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

print("Library berhasil diimport!")

## 2. Konfigurasi & Konstanta

In [ ]:
OUTPUT_DIR = r"D:\SEMESTER 4\KDA\project-kda-kelompok3\hasil"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 0 = Normal, 1 = Attack, 2 = Fault
LABEL_MAP   = {0: "Normal", 1: "Attack", 2: "Fault"}
LABEL_NAMES = ["Normal", "Attack", "Fault"]
N_ITERATIONS = 5

print(f"Output directory: {OUTPUT_DIR}")
print(f"Label mapping   : {LABEL_MAP}")
print(f"Jumlah iterasi  : {N_ITERATIONS}")

## 3. Load & Preprocessing Data

In [ ]:
def load_data():
    print("\n[1] Load data")
    train_df = pd.read_csv(r"D:\SEMESTER 4\KDA\project-kda-kelompok3\data\df_train.csv")
    test_df  = pd.read_csv(r"D:\SEMESTER 4\KDA\project-kda-kelompok3\data\df_test_lengkap.csv")

    drop_cols = ['timestamp', 'device_id']

    X_train  = train_df.drop(columns=drop_cols + ['label'], errors='ignore')
    y_train  = train_df['label']

    # Simpan df_test asli untuk dijadikan basis output
    test_raw = test_df.copy()

    # HIDE LABEL PADA TEST DATA: Drop label dari X_test dan simpan di y_test
    X_test   = test_df.drop(columns=drop_cols + ['label'], errors='ignore')
    y_test   = test_df['label']

    # Reset index
    y_train = y_train.reset_index(drop=True)
    y_test  = y_test.reset_index(drop=True)

    print(f"    Train : {X_train.shape[0]} baris | {X_train.shape[1]} fitur")
    print(f"    Test  : {X_test.shape[0]} baris  | {X_test.shape[1]} fitur")
    print(f"    Label : {LABEL_NAMES}")

    print("    Distribusi Train:")
    print(y_train.value_counts().sort_index().rename(LABEL_MAP).to_string())
    print("    Distribusi Test (Hidden untuk evaluasi):")
    print(y_test.value_counts().sort_index().rename(LABEL_MAP).to_string())

    scaler    = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    return X_train_s, y_train, X_test_s, y_test, test_raw


X_train, y_train, X_test, y_test, test_raw = load_data()

### Preview Data Training

In [ ]:
display(X_train.head())
print(f"\nShape X_train: {X_train.shape}")
print(f"Shape X_test : {X_test.shape}")

## 4. Definisi Model

In [ ]:
def get_models(seed=42):
    return {
        "Decision_Tree"      : DecisionTreeClassifier(random_state=seed, max_depth=10),
        "Random_Forest"      : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=seed, n_jobs=-1),
    }

print("Model yang akan dilatih:")
for name in get_models().keys():
    print(f"  - {name.replace('_', ' ')}")

## 5. Training & Prediksi (Multi-Iterasi)

In [ ]:
def train_and_predict_all(X_train, y_train, X_test, y_test, n_iterations=5):
    print(f"\n[2] Training setiap model ({n_iterations} iterasi masing-masing)...")
    print(f"    Model : Decision Tree | Random Forest | Logistic Regression\n")

    results   = {}
    pred_dict = {}

    for model_name in get_models().keys():
        print(f"  {'─'*50}")
        print(f"  Model: {model_name.replace('_', ' ')}")
        print(f"  {'─'*50}")

        all_proba    = []
        iter_records = []

        for i in range(n_iterations):
            seed  = 42 + i * 7
            model = get_models(seed=seed)[model_name]
            model.fit(X_train, y_train)

            proba = model.predict_proba(X_test)
            all_proba.append(proba)

            cv_scores = cross_val_score(
                get_models(seed=seed)[model_name],
                X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1
            )
            iter_records.append({
                'Model'      : model_name.replace('_', ' '),
                'Iterasi'    : i + 1,
                'Seed'       : seed,
                'CV_Acc_Mean': round(cv_scores.mean(), 4),
                'CV_Acc_Std' : round(cv_scores.std(),  4),
            })
            print(f"    Iter {i+1}/{n_iterations} (seed={seed}) → CV Acc: "
                  f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        # Agregasi probabilitas prediksi test dari semua iterasi
        avg_proba  = np.mean(all_proba, axis=0)
        final_pred = np.argmax(avg_proba, axis=1)

        # Evaluasi pada Train Set (menggunakan model final seed=42)
        final_model = get_models(seed=42)[model_name]
        final_model.fit(X_train, y_train)
        y_pred_train = final_model.predict(X_train)
        train_acc    = accuracy_score(y_train, y_pred_train)

        # Evaluasi pada Test Set
        test_acc = accuracy_score(y_test, final_pred)
        test_f1  = f1_score(y_test, final_pred, average='weighted')

        print(f"\n    Train Accuracy : {train_acc:.4f}")
        print(f"    Test Accuracy  : {test_acc:.4f}  |  Test F1 (weighted): {test_f1:.4f}")

        print(f"\n    Classification Report (Test Set):")
        report_names = [LABEL_MAP[i] for i in sorted(LABEL_MAP.keys())]
        for line in classification_report(
            y_test, final_pred,
            target_names=report_names,
            labels=sorted(LABEL_MAP.keys())
        ).splitlines():
            print(f"      {line}")

        results[model_name] = {
            'train_acc'   : train_acc,
            'test_acc'    : test_acc,
            'test_f1'     : test_f1,
            'cv_mean_last': iter_records[-1]['CV_Acc_Mean'],
            'iter_records': iter_records,
        }
        pred_dict[model_name] = {
            'pred'     : final_pred,
            'avg_proba': avg_proba,
        }

    return results, pred_dict


results, pred_dict = train_and_predict_all(
    X_train, y_train, X_test, y_test, n_iterations=N_ITERATIONS
)

## 6. Menyusun Output CSV

In [ ]:
def build_output_csv(test_raw, pred_dict):
    df = test_raw.copy().reset_index(drop=True)

    short = {
        "Decision_Tree"      : "DT",
        "Random_Forest"      : "RF",
        "Logistic_Regression": "LR",
    }

    for model_name, data in pred_dict.items():
        pfx  = short[model_name]
        pred = data['pred']
        df[f'{pfx}_prediction'] = pred

    return df


pred_df = build_output_csv(test_raw, pred_dict)
print("Output CSV berhasil disusun!")
display(pred_df.head())

## 7. Ringkasan Performa & Simpan Hasil

In [ ]:
summary_rows = []
for model_name, res in results.items():
    summary_rows.append({
        'Model'         : model_name.replace('_', ' '),
        'CV_Acc_Mean'   : f"{res['cv_mean_last']:.4f}",
        'Train_Accuracy': f"{res['train_acc']:.4f}",
        'Test_Accuracy' : f"{res['test_acc']:.4f}",
        'Test_F1'       : f"{res['test_f1']:.4f}",
    })
summary_df = pd.DataFrame(summary_rows)

all_iter = []
for res in results.values():
    all_iter.extend(res['iter_records'])
iter_df = pd.DataFrame(all_iter)

out_pred = os.path.join(OUTPUT_DIR, "test_predictions.csv")
out_summ = os.path.join(OUTPUT_DIR, "model_summary.csv")
out_iter = os.path.join(OUTPUT_DIR, "iteration_metrics.csv")

pred_df.to_csv(out_pred, index=False)
summary_df.to_csv(out_summ, index=False)
iter_df.to_csv(out_iter, index=False)

print("\n  RINGKASAN PERFORMA MODEL (PADA TEST SET)")
print("="*65)
display(summary_df)

### Distribusi Prediksi Test

In [ ]:
print("\n  DISTRIBUSI PREDIKSI TEST")
print("="*60)
for model_name, data in pred_dict.items():
    dist = pd.Series(data['pred']).map(LABEL_MAP).value_counts()
    print(f"  {model_name.replace('_',' '):<22}: {dict(dist)}")

### Preview `test_predictions.csv`

In [ ]:
cols_preview = ['timestamp', 'device_id', 'label', 'DT_prediction', 'RF_prediction', 'LR_prediction']
cols_preview = [c for c in cols_preview if c in pred_df.columns]
display(pred_df[cols_preview].head(10))

### Output Files

In [ ]:
print("  OUTPUT FILES:")
print("="*60)
print(f"  ├── test_predictions.csv  → df_test + kolom DT/RF/LR_prediction")
print(f"  ├── model_summary.csv     → Akurasi & F1 tiap model pada Test Set")
print(f"  └── iteration_metrics.csv → CV accuracy per iterasi per model")
print(f"\n  Tersimpan di: {OUTPUT_DIR}")

# Verifikasi file tersimpan
for f in [out_pred, out_summ, out_iter]:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✓ {os.path.basename(f)} ({size:,} bytes)")
    else:
        print(f"  ✗ {os.path.basename(f)} - TIDAK DITEMUKAN")